# Exp 1 — Batch-composition churn in detectors: breadth, mechanism, fix

Day 0 found that 38–46% of COCO images get different confident detections from DETR and Faster R-CNN depending on which images share their batch, while AP moves by only ±0.2. The no-padding control (RT-DETR) never changed.

This notebook turns that into paper-grade evidence in three parts. Each part saves its results as it finishes, so a crash late in the run does not lose earlier parts.

**Part A — Breadth.** Seven detectors across four design families on the same 500 images:
- DETR-style transformers: DETR, Deformable DETR;
- two-stage CNNs: Faster R-CNN, Mask R-CNN;
- one-stage CNNs: RetinaNet, FCOS;
- the no-padding control: RT-DETR.

Batching conditions are as in Day 0, plus a properly built **max-padding** condition. In it, every batch also contains the widest and the tallest image, so the padded canvas is always as large as possible.

**Part B — Mechanism** (200 images, one image at a time, padding added by hand):
- **B0, content test:** does it matter *what* the batch-mates show, or only how much padding they cause? We compare "batched with a real image" against "alone, padded to exactly the same canvas".
- **B1, dose and direction:** add 5 / 10 / 25 / 50% padding (right+bottom), and 25% right-only or bottom-only.
- **B2, where changes happen:** is a detection's chance of changing linked to its distance from the padded edge? If yes, the leak is *local* (convolution receptive fields). If no, it is *global* (attention spreads the change across the image).

**Part C — Fix.** Two ways to make batching invariant by construction:
- **fixed canvas:** always pad to the same size;
- **size buckets:** round each image's canvas up to a multiple of 128 and only batch images with equal canvases.

For each fix we measure invariance, AP, and speed against the default.

**Claims to confirm (fixed before running):**
- **C1:** at least 3 padding-based models flip ≥ 10% of images under random batching.
- **C2:** batch-mate *content* doesn't matter — B0 flip rate < 0.5%.
- **C3:** flip rate rises with padding amount for every Part-B model.
- **C4:** at least one fix gives ≤ 0.5% flips between its batch-1 and batch-8 runs, stays within 0.3 AP of default batch-1, and keeps ≥ 70% of default batch-8 speed.

**Kaggle settings:** GPU **T4 x2** or **P100**, Internet **ON**, nothing to attach. Expected runtime about 2–2.5 hours. If Kaggle stops the session, set `RUN_PARTS` in Section 1 to skip parts that already finished; their CSVs are in `/kaggle/working/exp1`.

In [ ]:
!pip install -q "transformers==5.17.0" pycocotools timm

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation
import io, sys, glob, json, time, math, random, zipfile, platform, datetime, contextlib, urllib.request, dataclasses, gc
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torchvision
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import transformers
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

OUT = "/kaggle/working/exp1"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

ENV = {
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if ENV["gpu"] is None:
    print("\n⚠⚠ NO GPU. Stop, set Accelerator to GPU in the right panel, and Run All again. ⚠⚠")

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

## 1. Configuration

In [ ]:
CFG = {
    "n_images": 500,
    "seed": 0,
    "batch_size": 8,
    "random_seeds": [0, 1, 2],
    "models_A": ["detr_r50", "deformable_detr", "frcnn_v2", "maskrcnn_v2", "retinanet_v2", "fcos", "rtdetr_r50"],
    "control_models": ["rtdetr_r50"],
    "models_B": ["detr_r50", "deformable_detr", "frcnn_v2"],
    "n_images_B": 200,
    "pad_levels_B": [0.05, 0.10, 0.25, 0.50],
    "models_C": ["detr_r50", "frcnn_v2"],
    "bucket": 128,
    "fixed_canvas": {"hf": (1333, 1333), "tv": (1344, 1344)},
    "flip_score_thr": 0.5,
    "flip_iou": 0.9,
    # Largest piece that fits in a 15 GB T4 for memory-hungry models. Pieces are padded to the full batch's
    # canvas, which gives the same result as the full batch (see MicroBatch in Section 3).
    "micro_batch": {"deformable_detr": 2},
    "coco_root": "/kaggle/input",
    "coco_download_dir": "/kaggle/working/coco",
    "claims": {"C1_flip": 0.10, "C1_models": 3, "C2_flip": 0.005, "C4_flip": 0.005, "C4_dap": 0.3, "C4_speed": 0.70},
}
RUN_PARTS = {"A": True, "B": True, "C": True}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

## 2. COCO val2017 (same 500 images as Day 0)

In [ ]:
def find_attached_coco(root):
    anns = glob.glob(os.path.join(root, "**", "instances_val2017.json"), recursive=True)
    probe = glob.glob(os.path.join(root, "**", "000000000139.jpg"), recursive=True)
    if anns and probe:
        return anns[0], os.path.dirname(probe[0])
    return None, None

ANN, IMG_DIR = find_attached_coco(CFG["coco_root"])
if ANN is None:
    os.makedirs(CFG["coco_download_dir"], exist_ok=True)
    ANN = os.path.join(CFG["coco_download_dir"], "instances_val2017.json")
    IMG_DIR = os.path.join(CFG["coco_download_dir"], "val2017")
    if not os.path.exists(ANN):
        zp = os.path.join(CFG["coco_download_dir"], "ann.zip")
        print("downloading COCO annotations (~250 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp) as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
        os.remove(zp)

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(ANN)
NAME2CAT = {c["name"].strip().lower(): c["id"] for c in GT.loadCats(GT.getCatIds())}
IDS = sorted(random.Random(CFG["seed"]).sample(sorted(GT.getImgIds()), CFG["n_images"]))
INFO = GT.loadImgs(IDS)
os.makedirs(IMG_DIR, exist_ok=True)
def fetch(info):
    p = os.path.join(IMG_DIR, info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{info['file_name']}", p)
    return p
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, INFO))
IMGS = [Image.open(p).convert("RGB") for p in paths]
N = len(IMGS)
print(f"{N} images; first ids {IDS[:3]} (Day 0 used seed 0 too, so these are the same images)")

## 3. Detectors

One interface for all models. `run(images, canvas=None)`:
- **`canvas=None`** is the library's normal batching (pad to the largest image in the batch).
- **`canvas=(H, W)`** pads every image to exactly that size in the model's input space. This is how Parts B and C control padding by hand.

A self-test in Section 5 checks that the two modes agree when the canvas equals the batch's own canvas.

Class names are mapped to COCO ids with a small alias table (Day 0 lost 6 RT-DETR classes to spelling differences). Any name that still fails to map is printed.

In [ ]:
ALIASES = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch", "tvmonitor": "tv", "tv monitor": "tv",
           "diningtable": "dining table", "pottedplant": "potted plant", "hair dryer": "hair drier",
           "sports ball": "sports ball", "cellphone": "cell phone", "mobile phone": "cell phone"}
def _norm(s):
    s = " ".join(str(s).lower().replace("_", " ").replace("-", " ").split())
    return ALIASES.get(s, s)
def names_to_cat(names, label=""):
    m, miss = {}, []
    for k, v in names.items():
        c = NAME2CAT.get(_norm(v))
        if c is not None:
            m[int(k)] = c
        elif _norm(v) not in ("n/a", "__background__", "background", "no object"):
            miss.append(v)
    if miss:
        print(f"  unmapped class names in {label}: {miss}")
    return m

def load_hf_model(cls, repo, revision=None):
    try:
        return cls.from_pretrained(repo, revision=revision)
    except Exception as ex:
        if "alidation" not in f"{type(ex).__name__} {ex}":
            raise
        from huggingface_hub import hf_hub_download
        raw = json.load(open(hf_hub_download(repo, "config.json", revision=revision)))
        defaults = {f.name: f.default for f in dataclasses.fields(cls.config_class) if f.default is not dataclasses.MISSING}
        dropped = [k for k, v in raw.items() if v is None and defaults.get(k) is not None]
        print(f"  (config fix for {repo}: replaced null fields {dropped} with library defaults)")
        raw = {k: v for k, v in raw.items() if k not in dropped}
        return cls.from_pretrained(repo, revision=revision, config=cls.config_class.from_dict(raw))

def roundup(x, d):
    return int(math.ceil(x / d) * d)

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

def _pack(boxes, scores, labels, lab2cat):
    lab = labels.cpu().numpy()
    keep = np.array([l in lab2cat for l in lab], bool)
    return {"boxes": boxes.float().cpu().numpy()[keep], "scores": scores.float().cpu().numpy()[keep],
            "labels": np.array([lab2cat[l] for l in lab[keep]], dtype=int)}

class MicroBatch:
    """If a batch does not fit in GPU memory, split it into smaller pieces that are all padded to the canvas the
    full batch would have had. Images in these models never interact except through padding, so this gives the
    same result as the full batch (verified for DETR by the canvas self-test, and for every Part-B model by B0)."""
    micro = None   # largest piece size known to fit; None = no limit found yet

    def batch_canvas(self, imgs):
        sizes = [self.input_size(im) for im in imgs]
        return (roundup(max(h for h, _ in sizes), self.div), roundup(max(w for _, w in sizes), self.div))

    def run(self, imgs, canvas=None):
        if self.fixed_size:
            return self._run(imgs, canvas)
        if self.micro is not None and len(imgs) > self.micro:
            return self._split(imgs, canvas)
        try:
            return self._run(imgs, canvas)
        except torch.cuda.OutOfMemoryError:
            if len(imgs) == 1:
                raise
        # We are now OUTSIDE the except block, so the failed attempt (and the GPU memory it holds) is released
        # before retrying. Retrying inside the except block keeps that memory alive.
        free_mem()
        self.micro = max(1, len(imgs) // 2)
        print(f"  [{self.name}] batch of {len(imgs)} does not fit in GPU memory → running pieces of {self.micro} "
              f"on the full batch's canvas (equivalent result)")
        return self._split(imgs, canvas)

    def _split(self, imgs, canvas):
        canvas = canvas if canvas is not None else self.batch_canvas(imgs)
        step = self.micro                      # fixed for this call, even if a piece lowers self.micro further
        R, P = [], []
        for k in range(0, len(imgs), step):
            r, p = self.run(imgs[k:k + step], canvas)
            R += r; P += p
        assert len(R) == len(imgs)
        return R, P

class HFDet(MicroBatch):
    kind, div = "hf", 1
    def __init__(self, name, repo, cls_name, revision=None, fixed_size=False):
        from transformers import AutoImageProcessor
        self.name, self.fixed_size = name, fixed_size
        self.proc = AutoImageProcessor.from_pretrained(repo, revision=revision)
        self.model = load_hf_model(getattr(transformers, cls_name), repo, revision).to(DEVICE).eval()
        self.lab2cat = names_to_cat(self.model.config.id2label, name)

    def _single(self, img):
        enc = self.proc(images=[img], return_tensors="pt")
        pv = enc["pixel_values"][0]
        m = enc["pixel_mask"][0] if "pixel_mask" in enc else torch.ones(pv.shape[-2:], dtype=torch.long)
        return pv, m

    def input_size(self, img):
        return tuple(self._single(img)[0].shape[-2:])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        if canvas is None:
            enc = self.proc(images=imgs, return_tensors="pt")
            pv = enc["pixel_values"]
            mask = enc["pixel_mask"] if "pixel_mask" in enc else None
        else:
            H, W = canvas; pvs, ms = [], []
            for im in imgs:
                p, m = self._single(im); h, w = p.shape[-2:]
                assert h <= H and w <= W, f"canvas {canvas} smaller than image {(h, w)}"
                pvs.append(F.pad(p, (0, W - w, 0, H - h))); ms.append(F.pad(m, (0, W - w, 0, H - h)))
            pv, mask = torch.stack(pvs), torch.stack(ms)
        kw = {"pixel_values": pv.to(DEVICE)}
        if mask is not None:
            kw["pixel_mask"] = mask.to(DEVICE)
        out = self.model(**kw)
        sizes = torch.tensor([[im.size[1], im.size[0]] for im in imgs])
        res = self.proc.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        pad = (1 - mask.float().mean(dim=(1, 2))).tolist() if mask is not None else [0.0] * len(imgs)
        return [_pack(r["boxes"], r["scores"], r["labels"], self.lab2cat) for r in res], pad

class TVDet(MicroBatch):
    kind, div, fixed_size = "tv", 32, False
    def __init__(self, name, builder, weights):
        self.name = name
        self.model = builder(weights=weights).to(DEVICE).eval()
        self.tf = weights.transforms()
        self.lab2cat = names_to_cat(dict(enumerate(weights.meta["categories"])), name)
        self._canvas = None
        orig = self.model.transform.batch_images
        def batch_images(images, size_divisible=32):
            b = orig(images, size_divisible)
            if self._canvas is None:
                return b
            H, W = self._canvas
            assert b.shape[-2] <= H and b.shape[-1] <= W, f"canvas {self._canvas} smaller than batch {tuple(b.shape[-2:])}"
            return F.pad(b, (0, W - b.shape[-1], 0, H - b.shape[-2]))
        self.model.transform.batch_images = batch_images

    @torch.inference_mode()
    def input_size(self, img):
        il, _ = self.model.transform([self.tf(img).to(DEVICE)])
        return tuple(il.image_sizes[0])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        self._canvas = canvas
        try:
            x = [self.tf(im).to(DEVICE) for im in imgs]
            il, _ = self.model.transform(x)
            H, W = il.tensors.shape[-2:]
            pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
            out = self.model(x)
        finally:
            self._canvas = None
        return [_pack(o["boxes"], o["scores"], o["labels"], self.lab2cat) for o in out], pad

def _tv(builder_name, weights_name, name):
    import torchvision.models.detection as D
    return lambda: TVDet(name, getattr(D, builder_name), getattr(D, weights_name).DEFAULT)

LOADERS = {
    "detr_r50": lambda: HFDet("detr_r50", "facebook/detr-resnet-50", "DetrForObjectDetection", revision="no_timm"),
    "deformable_detr": lambda: HFDet("deformable_detr", "SenseTime/deformable-detr", "DeformableDetrForObjectDetection"),
    "frcnn_v2": _tv("fasterrcnn_resnet50_fpn_v2", "FasterRCNN_ResNet50_FPN_V2_Weights", "frcnn_v2"),
    "maskrcnn_v2": _tv("maskrcnn_resnet50_fpn_v2", "MaskRCNN_ResNet50_FPN_V2_Weights", "maskrcnn_v2"),
    "retinanet_v2": _tv("retinanet_resnet50_fpn_v2", "RetinaNet_ResNet50_FPN_V2_Weights", "retinanet_v2"),
    "fcos": _tv("fcos_resnet50_fpn", "FCOS_ResNet50_FPN_Weights", "fcos"),
    "rtdetr_r50": lambda: HFDet("rtdetr_r50", "PekingU/rtdetr_r50vd", "RTDetrForObjectDetection", fixed_size=True),
}
# Published full COCO val2017 box AP; our 500-image batch-1 AP should be within about ±2.
REF_AP = {"detr_r50": 42.0, "deformable_detr": 44.5, "frcnn_v2": 46.7, "maskrcnn_v2": 47.4,
          "retinanet_v2": 41.5, "fcos": 39.2, "rtdetr_r50": 53.1}

def load(name):
    free_mem()
    t0 = time.time()
    try:
        det = LOADERS[name]()
        det.micro = CFG["micro_batch"].get(name)
    except Exception as ex:
        print(f"✗ {name} failed to load: {type(ex).__name__}: {str(ex)[:300]}  — skipping this model")
        return None
    note = f"; runs in pieces of {det.micro} (padded to the full batch's canvas)" if det.micro else ""
    print(f"\n== {name} loaded in {time.time() - t0:.0f}s; mapped classes {len(set(det.lab2cat.values()))}/80{note}")
    return det

def free_mem():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 4. Metrics

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(p, q, thr, iou_min=0.5):
    ia = np.where(p["scores"] >= thr)[0]; ib = np.where(q["scores"] >= thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bc = -1.0, None
        for c, j in enumerate(ib):
            if c not in used and p["labels"][i] == q["labels"][j] and M[r, c] > best:
                best, bc = M[r, c], c
        if bc is not None and best >= iou_min:
            used.add(bc); pairs.append((i, ib[bc], best))
    return pairs, ia, ib

def compare(ref, new):
    """Image-level flip + detection-level change counts + max score change, ref vs new."""
    pairs, ia, ib = greedy_match(ref, new, CFG["flip_score_thr"])
    stable = sum(iou >= CFG["flip_iou"] for _, _, iou in pairs)
    moved = len(pairs) - stable
    lost, gained = len(ia) - len(pairs), len(ib) - len(pairs)
    allp, _, _ = greedy_match(ref, new, 0.05)
    dscore = max([abs(ref["scores"][i] - new["scores"][j]) for i, j, _ in allp], default=0.0)
    return {"flip": (moved + lost + gained) > 0, "stable": stable, "moved": moved, "lost": lost, "gained": gained,
            "max_score_delta": dscore}

def det_survival(ref, new, W, H):
    """For each confident ref detection: did it survive unchanged? plus its distance to the right/bottom edge."""
    pairs, ia, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ok = {i for i, _, iou in pairs if iou >= CFG["flip_iou"]}
    rows = []
    for i in ia:
        x1, y1, x2, y2 = ref["boxes"][i]
        rows.append({"changed": i not in ok, "d_right": (W - x2) / W, "d_bottom": (H - y2) / H,
                     "rel_area": (x2 - x1) * (y2 - y1) / (W * H), "score": ref["scores"][i]})
    return rows

def coco_ap(preds, idx=None):
    idx = list(range(N)) if idx is None else idx
    dets = []
    for k in idx:
        r = preds[k]; order = np.argsort(-r["scores"])[:100]
        for t in order:
            x1, y1, x2, y2 = r["boxes"][t]
            dets.append({"image_id": IDS[k], "category_id": int(r["labels"][t]),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(r["scores"][t])})
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "bbox"); E.params.imgIds = [IDS[k] for k in idx]
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

def run_batches(det, batches, canvas_fn=None, targets=None):
    """batches: list of index lists. canvas_fn(batch)->(H,W) or None. targets: optional list of index lists
    saying which members of each batch to record (others are companions). Returns preds, pads, seconds."""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([IMGS[i] for i in b], canvas=canvas_fn(b) if canvas_fn else None)
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

## 5. Batching plans + self-test

In [ ]:
B = CFG["batch_size"]
def chunks(order, k=B):
    order = list(order)
    return [order[i:i + k] for i in range(0, len(order), k)]

aspect = np.array([im.size[0] / im.size[1] for im in IMGS])
area = np.array([im.size[0] * im.size[1] for im in IMGS])
WIDEST, TALLEST = int(np.argmax(aspect)), int(np.argmin(aspect))
PLANS = {"bs1": ([[i] for i in range(N)], None), "bs1_repeat": ([[i] for i in range(N)], None)}
for s in CFG["random_seeds"]:
    PLANS[f"bs8_rand_s{s}"] = (chunks(np.random.default_rng(s).permutation(N)), None)
PLANS["bs8_sorted"] = (chunks(np.lexsort((area, aspect))), None)
mp = chunks(range(N), B - 2)
PLANS["bs8_maxpad"] = ([c + [WIDEST, TALLEST] for c in mp], mp)     # anchors force the largest canvas; only targets recorded
print("widest image aspect", round(aspect[WIDEST], 2), "| tallest", round(aspect[TALLEST], 2))
print({k: len(v[0]) for k, v in PLANS.items()})

def roundup(x, d):
    return int(math.ceil(x / d) * d)

def self_test(det, k=16):
    """canvas-mode with the batch's own canvas must reproduce default batching."""
    idx = list(range(min(k, N)))
    sizes = [det.input_size(IMGS[i]) for i in idx]
    H = roundup(max(s[0] for s in sizes), det.div); W = roundup(max(s[1] for s in sizes), det.div)
    r1, _ = det.run([IMGS[i] for i in idx[:8]]); r2, _ = det.run([IMGS[i] for i in idx[:8]], canvas=None)
    sz8 = sizes[:8]; c8 = (roundup(max(s[0] for s in sz8), det.div), roundup(max(s[1] for s in sz8), det.div))
    r3, _ = det.run([IMGS[i] for i in idx[:8]], canvas=c8)
    d_rep = max(compare(a, b)["max_score_delta"] for a, b in zip(r1, r2))
    d_can = max(compare(a, b)["max_score_delta"] for a, b in zip(r1, r3))
    f_can = sum(compare(a, b)["flip"] for a, b in zip(r1, r3))
    note = f" (note: GPU memory forced pieces of {det.micro}, so this compares like with like)" if det.micro else ""
    print(f"  self-test: repeat max|Δscore|={d_rep:.2e}; canvas-mode vs default max|Δscore|={d_can:.2e}, flips={f_can}/8{note}")
    return d_can < 1e-3 and f_can == 0

## 6. Part A — Breadth across detector families

In [ ]:
if RUN_PARTS["A"]:
    A_res, A_img = [], []
    if os.path.exists(f"{OUT}/A_ap.csv"):
        A_res = read_csv(f"{OUT}/A_ap.csv").to_dict("records")
        A_img = read_csv(f"{OUT}/A_per_image.csv").to_dict("records")
    doneA = {r["model"] for r in A_res}
    for mname in CFG["models_A"]:
        if mname in doneA:
            print(f"== {mname}: already done (found in A_ap.csv), skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None:
            continue
        if not det.fixed_size:
            ok = self_test(det)
            print("  ✓ canvas mode reproduces default batching" if ok else "  ⚠ canvas mode differs from default batching (Parts B/C less reliable for this model)")
        preds = {}
        for cname, (batches, targets) in PLANS.items():
            P, PD, sec = run_batches(det, batches, targets=targets)
            preds[cname] = P
            ap = coco_ap(P)
            A_res.append({"model": mname, "condition": cname, "AP": ap, "mean_pad": float(np.mean(PD)), "sec_per_img": sec / N})
            ref = f"  (published ≈ {REF_AP[mname]})" if cname == "bs1" else ""
            print(f"  {cname:13s} AP={ap:6.2f}  pad={np.mean(PD):.3f}  {1000 * sec / N:5.1f} ms/img{ref}")
            if cname != "bs1":
                for i in range(N):
                    A_img.append({"model": mname, "condition": cname, "image_id": IDS[i], "pad": PD[i],
                                  **compare(preds["bs1"][i], P[i])})
        det = None; preds = None; free_mem()
        pd.DataFrame(A_res).to_csv(f"{OUT}/A_ap.csv", index=False)
        pd.DataFrame(A_img).to_csv(f"{OUT}/A_per_image.csv", index=False)

In [ ]:
A_res = read_csv(f"{OUT}/A_ap.csv"); A_img = read_csv(f"{OUT}/A_per_image.csv")
base = A_res[A_res.condition == "bs1"].set_index("model")["AP"]
A_res["dAP"] = A_res.apply(lambda r: r.AP - base[r.model], axis=1)
agg = A_img.groupby(["model", "condition"]).agg(
    flip_rate=("flip", "mean"), moved=("moved", "sum"), lost=("lost", "sum"), gained=("gained", "sum"),
    stable=("stable", "sum"), p95_dscore=("max_score_delta", lambda v: np.percentile(v, 95))).reset_index()
agg["det_change_rate"] = (agg.moved + agg.lost + agg.gained) / (agg.stable + agg.moved + agg.lost + agg.gained).clip(lower=1)
TA = A_res.merge(agg, on=["model", "condition"], how="left")
TA.to_csv(f"{OUT}/A_summary.csv", index=False)
show = TA[TA.condition.isin(["bs1", "bs8_rand_s0", "bs8_sorted", "bs8_maxpad"])]
display(show[["model", "condition", "AP", "dAP", "flip_rate", "det_change_rate", "p95_dscore", "mean_pad", "sec_per_img"]]
        .style.format({"AP": "{:.2f}", "dAP": "{:+.2f}", "flip_rate": "{:.1%}", "det_change_rate": "{:.1%}",
                       "p95_dscore": "{:.3f}", "mean_pad": "{:.2f}", "sec_per_img": "{:.3f}"}, na_rep="—"))

rand = TA[TA.condition.str.startswith("bs8_rand")].groupby("model").agg(
    flip=("flip_rate", "mean"), det_change=("det_change_rate", "mean"), AP_min=("AP", "min"), AP_max=("AP", "max"))
rand["seed_AP_range"] = rand.AP_max - rand.AP_min
print("Random batch-8 (mean over 3 shuffles):"); display(rand.style.format("{:.3f}"))

fig, ax = plt.subplots(figsize=(11, 4))
order = [m for m in CFG["models_A"] if m in rand.index]
conds = ["bs8_sorted", "bs8_rand_s0", "bs8_maxpad"]
w = 0.27
for k, c in enumerate(conds):
    v = agg[agg.condition == c].set_index("model").reindex(order)["flip_rate"] * 100
    ax.bar(np.arange(len(order)) + (k - 1) * w, v, w, label=c)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=20)
ax.set_ylabel("% images with changed confident detections"); ax.legend(); ax.set_title("Part A — flip rate vs batch-1, by model")
plt.tight_layout(); plt.savefig(f"{OUT}/A_flip_rates.png", dpi=180); plt.show()

## 7. Part B — Mechanism (one image at a time, hand-made padding)

In [ ]:
NB = CFG["n_images_B"]
if RUN_PARTS["B"]:
    rng = np.random.default_rng(CFG["seed"] + 1)
    B_img, B_det = [], []
    if os.path.exists(f"{OUT}/B_per_image.csv"):
        B_img = read_csv(f"{OUT}/B_per_image.csv").to_dict("records")
        B_det = read_csv(f"{OUT}/B_per_detection.csv").to_dict("records")
    doneB = {r["model"] for r in B_img}
    for mname in CFG["models_B"]:
        if mname in doneB:
            print(f"== {mname}: Part B already done, skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None or det.fixed_size:
            continue
        sizes = [det.input_size(IMGS[i]) for i in range(NB)]
        t0 = time.time()
        for i in range(NB):
            h, w = sizes[i]
            W0, H0 = IMGS[i].size
            base, _ = det.run([IMGS[i]])
            base = base[0]
            # B0: real companion vs. blank padding to the identical canvas
            j = int(rng.choice([k for k in range(N) if k != i]))
            hj, wj = det.input_size(IMGS[j])
            canvas = (roundup(max(h, hj), det.div), roundup(max(w, wj), det.div))
            with_comp, _ = det.run([IMGS[i], IMGS[j]])
            alone_same, _ = det.run([IMGS[i]], canvas=canvas)
            c = compare(with_comp[0], alone_same[0])
            B_img.append({"model": mname, "test": "B0_content", "level": 0, "image_idx": i, **c})
            # B1: dose (right+bottom) and direction
            tests = [("both", p) for p in CFG["pad_levels_B"]] + [("right", 0.25), ("bottom", 0.25)]
            for direction, p in tests:
                Hc = roundup(h * (1 + p) if direction in ("both", "bottom") else h, det.div)
                Wc = roundup(w * (1 + p) if direction in ("both", "right") else w, det.div)
                r, pad = det.run([IMGS[i]], canvas=(Hc, Wc))
                c = compare(base, r[0])
                B_img.append({"model": mname, "test": f"B1_{direction}", "level": p, "image_idx": i, "pad": pad[0], **c})
                for d in det_survival(base, r[0], W0, H0):
                    B_det.append({"model": mname, "test": f"B1_{direction}", "level": p, **d})
            if i % 50 == 0:
                print(f"  {mname}: {i + 1}/{NB} images  ({time.time() - t0:.0f}s)")
        det = None; free_mem()
        pd.DataFrame(B_img).to_csv(f"{OUT}/B_per_image.csv", index=False)
        pd.DataFrame(B_det).to_csv(f"{OUT}/B_per_detection.csv", index=False)

In [ ]:
B_img = read_csv(f"{OUT}/B_per_image.csv"); B_det = read_csv(f"{OUT}/B_per_detection.csv")
b0 = B_img[B_img.test == "B0_content"].groupby("model").agg(flip_rate=("flip", "mean"), max_dscore=("max_score_delta", "max"))
print("B0 — real batch-mate vs blank padding to the same canvas (0% = content doesn't matter):")
display(b0.style.format({"flip_rate": "{:.2%}", "max_dscore": "{:.2e}"}))

b1 = B_img[B_img.test.str.startswith("B1")].groupby(["model", "test", "level"]).agg(
    flip_rate=("flip", "mean"), mean_pad=("pad", "mean")).reset_index()
print("B1 — flip rate vs added padding:")
display(b1.pivot_table(index=["test", "level"], columns="model", values="flip_rate").style.format("{:.1%}"))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for m in b1.model.unique():
    d = b1[(b1.model == m) & (b1.test == "B1_both")]
    axes[0].plot(d.level * 100, d.flip_rate * 100, "o-", label=m)
axes[0].set_xlabel("padding added on right and bottom (% of each side)"); axes[0].set_ylabel("flip rate (%)")
axes[0].set_title("B1 — dose response"); axes[0].legend()
bins = [0, 0.05, 0.1, 0.2, 0.35, 0.5, 1.01]
for ax, direction, col in [(axes[1], "B1_right", "d_right"), (axes[2], "B1_bottom", "d_bottom")]:
    for m in B_det.model.unique():
        d = B_det[(B_det.model == m) & (B_det.test == direction)].copy()
        if d.empty:
            continue
        d["bin"] = pd.cut(d[col], bins, right=False)
        g = d.groupby("bin", observed=True)["changed"].mean() * 100
        ax.plot([f"{b.left:.2f}" for b in g.index], g.values, "o-", label=m)
    ax.set_xlabel(f"detection's distance to the padded {direction.split('_')[1]} edge (fraction of image)")
    ax.set_ylabel("% detections changed"); ax.set_title(f"B2 — {direction.split('_')[1]}-only padding (25%)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/B_mechanism.png", dpi=180); plt.show()

**How to read B2.** A falling curve (changes concentrated near the padded edge) means a *local* leak through convolution receptive fields. A flat curve means a *global* leak: the change spreads to the whole image, e.g. through attention.

## 8. Part C — Invariant-by-construction fixes

In [ ]:
if RUN_PARTS["C"]:
    C_res, C_img = [], []
    if os.path.exists(f"{OUT}/C_runs.csv"):
        C_res = read_csv(f"{OUT}/C_runs.csv").to_dict("records")
        C_img = read_csv(f"{OUT}/C_per_image.csv").to_dict("records")
    doneC = {r["model"] for r in C_res}
    for mname in CFG["models_C"]:
        if mname in doneC:
            print(f"== {mname}: Part C already done, skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None or det.fixed_size:
            continue
        sizes = [det.input_size(IMGS[i]) for i in range(N)]
        fixed = CFG["fixed_canvas"][det.kind]
        bucket = [(roundup(h, CFG["bucket"]), roundup(w, CFG["bucket"])) for h, w in sizes]
        groups = {}
        for i, c in enumerate(bucket):
            groups.setdefault(c, []).append(i)
        bucket_batches = [b for g in groups.values() for b in chunks(g)]
        rand0 = chunks(np.random.default_rng(0).permutation(N))
        runs = {
            "default_bs1": ([[i] for i in range(N)], None),
            "default_bs8_rand": (rand0, None),
            "fixed_bs1": ([[i] for i in range(N)], lambda b: fixed),
            "fixed_bs8_rand": (rand0, lambda b: fixed),
            "bucket_bs1": ([[i] for i in range(N)], lambda b: bucket[b[0]]),
            "bucket_bs8": (bucket_batches, lambda b: bucket[b[0]]),
        }
        print(f"  {len(groups)} size buckets; batches in bucket plan: {len(bucket_batches)} (vs {len(rand0)} random)")
        preds = {}
        for rname, (batches, cf) in runs.items():
            try:
                P, PD, sec = run_batches(det, batches, canvas_fn=cf)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print(f"  {rname}: out of GPU memory at batch 8, retrying with batch 4")
                batches = [c for b in batches for c in chunks(b, 4)]
                P, PD, sec = run_batches(det, batches, canvas_fn=cf)
            preds[rname] = P
            ap = coco_ap(P)
            C_res.append({"model": mname, "run": rname, "AP": ap, "sec_per_img": sec / N, "mean_pad": float(np.mean(PD))})
            print(f"  {rname:17s} AP={ap:6.2f}  pad={np.mean(PD):.3f}  {1000 * sec / N:5.1f} ms/img")
        for a, b in [("default_bs1", "default_bs8_rand"), ("fixed_bs1", "fixed_bs8_rand"), ("bucket_bs1", "bucket_bs8"),
                     ("default_bs1", "fixed_bs1"), ("default_bs1", "bucket_bs1")]:
            for i in range(N):
                C_img.append({"model": mname, "pair": f"{a} vs {b}", **compare(preds[a][i], preds[b][i])})
        det = None; preds = None; free_mem()
        pd.DataFrame(C_res).to_csv(f"{OUT}/C_runs.csv", index=False)
        pd.DataFrame(C_img).to_csv(f"{OUT}/C_per_image.csv", index=False)

In [ ]:
C_res = read_csv(f"{OUT}/C_runs.csv"); C_img = read_csv(f"{OUT}/C_per_image.csv")
cb = C_res[C_res.run == "default_bs1"].set_index("model")
C_res["dAP_vs_default_bs1"] = C_res.apply(lambda r: r.AP - cb.loc[r.model, "AP"], axis=1)
C_res["speed_vs_default_bs8"] = C_res.apply(
    lambda r: C_res[(C_res.model == r.model) & (C_res.run == "default_bs8_rand")].sec_per_img.iloc[0] / r.sec_per_img, axis=1)
display(C_res.style.format({"AP": "{:.2f}", "sec_per_img": "{:.4f}", "mean_pad": "{:.2f}",
                            "dAP_vs_default_bs1": "{:+.2f}", "speed_vs_default_bs8": "{:.2f}×"}))
cf = C_img.groupby(["model", "pair"]).agg(flip_rate=("flip", "mean")).reset_index()
print("Flip rate between runs (invariance check: 'fixed_bs1 vs fixed_bs8_rand' and 'bucket_bs1 vs bucket_bs8' should be ~0%):")
display(cf.pivot(index="pair", columns="model", values="flip_rate").style.format("{:.2%}"))

## 9. Claims check

In [ ]:
K = CFG["claims"]; verdicts = {}
pad_models = [m for m in rand.index if m not in CFG["control_models"]]
n_hi = int((rand.loc[pad_models, "flip"] >= K["C1_flip"]).sum())
verdicts["C1_breadth"] = f"{'✓' if n_hi >= K['C1_models'] else '✗'}  {n_hi}/{len(pad_models)} padding models flip ≥{K['C1_flip']:.0%}"
if len(b0):
    ok = (b0.flip_rate < K["C2_flip"]).all()
    verdicts["C2_content_independent"] = f"{'✓' if ok else '✗ (batch-mate content matters — report as its own finding)'}  max B0 flip {b0.flip_rate.max():.2%}"
mono = []
for m in b1.model.unique():
    v = b1[(b1.model == m) & (b1.test == "B1_both")].sort_values("level").flip_rate.values
    mono.append(bool(np.all(np.diff(v) >= -0.01)))
verdicts["C3_dose_response"] = f"{'✓' if mono and all(mono) else '✗'}  monotone for {sum(mono)}/{len(mono)} models"
fix_ok = []
for m in C_res.model.unique():
    for fx, (a, b) in {"fixed": ("fixed_bs1", "fixed_bs8_rand"), "bucket": ("bucket_bs1", "bucket_bs8")}.items():
        inv = cf[(cf.model == m) & (cf.pair == f"{a} vs {b}")].flip_rate.iloc[0]
        r = C_res[(C_res.model == m) & (C_res.run == b)].iloc[0]
        good = inv <= K["C4_flip"] and abs(r.dAP_vs_default_bs1) <= K["C4_dap"] and r.speed_vs_default_bs8 >= K["C4_speed"]
        fix_ok.append(good)
        print(f"  {m:10s} {fx:6s}: invariance flips {inv:.2%}, ΔAP {r.dAP_vs_default_bs1:+.2f}, speed {r.speed_vs_default_bs8:.2f}× of default bs8 → {'OK' if good else 'not OK'}")
verdicts["C4_fix"] = f"{'✓' if any(fix_ok) else '✗'}  {sum(fix_ok)}/{len(fix_ok)} model×fix combos meet all three criteria"
for k, v in verdicts.items():
    print(f"{k:25s} {v}")

with open(f"{OUT}/manifest.json", "w") as f:
    json.dump({"env": ENV, "config": CFG, "image_ids": IDS, "verdicts": verdicts}, f, indent=2, default=str)
print("saved:", sorted(os.listdir(OUT)))

## What to bring back

Upload the finished notebook. The important outputs are the Part A table and figure, the B0 line, the B1 table, the mechanism figure, the Part C tables, and the four claim lines at the end.